In [2]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import preprocessing as pp
import candidates as ca
import refinements as rf
import subgroup_selection as ss
import search as bs

In [3]:
unclean_data = pd.read_csv('algebrakit_firstbatch_2.csv')
df = pp.clean_data_algebrakit(unfilterd_data=unclean_data, filter_type="remove_interaction")
print(f"we have {len(df)} records of {len(df['sessionID'].unique())} sessions with {df['finished'].value_counts()}")

we have 6020 records of 1622 sessions with finished
False    3958
True     2062
Name: count, dtype: int64


In [5]:
df.head()

,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,...,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.00,4.0,Q1,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.00,4.0,Q2,...,0.0,0.00,1.0,False,2.0,2.0,True,0.00,1,2
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,True,1744026958,[FractionArithmeticMultiplicationProper],[CALCULATE],True,0.65,4.0,Q1,...,1.0,0.65,1.0,True,2.0,1.0,False,0.35,1,1
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.00,4.0,Q1,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.00,4.0,Q2,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0


In [34]:
df_onehot = df.copy()
pd.set_option('display.max_columns', None)


collection_attr = ['skills', 'positive_skills', 'negative_skills']

for attr in collection_attr:

    df_attr = df_onehot[[attr]]

    # Step 2: One-hot encoding with empty list handling
    attr_onehot = pd.get_dummies(df_attr[attr].apply(pd.Series).stack(), prefix=attr).groupby(level=0).sum()
        
    # Ensure rows with empty lists are handled
    df_attr_onehot = df_attr.drop(attr, axis=1).join(attr_onehot, how='left').fillna(0).astype(bool)

    df_onehot = df_onehot.drop(attr, axis=1)
    df_onehot = pd.concat([df_onehot, df_attr_onehot], axis=1)

df_onehot

,exerciseID,sessionID,success,creationTimestamp,stepTypes,finished,marksEarned,marksTotal,questionID,questionType,interactionID,itemType,interactionType,solution,content,event_types,event_evaluations,result_status,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors,skills_AlgebraAddingLikeTerms,skills_AlgebraProductOfPowerFunctions,skills_ArithmeticOrderDivision,skills_ArithmeticOrderMultiplication,skills_EquationsBalanceMethodArrangeTerms,skills_EquationsBalanceMethodDivideCoefficient,skills_EquationsOperationsAdd,skills_EquationsOperationsMultiply,skills_ExpandSingleBrackets,skills_FactorSingleBrackets,skills_FractionArithmeticAddProperCommonDenominator,skills_FractionArithmeticAddProperUnlikeDenominator,skills_FractionArithmeticDivisionProper,skills_FractionArithmeticEquivalentConcept,skills_FractionArithmeticMultiplicationMixed,skills_FractionArithmeticMultiplicationProper,skills_FractionArithmeticSimilarImproperToMixed,skills_FractionArithmeticSimilarMixed,skills_FractionArithmeticSimilarMixedToImproper,skills_FractionArithmeticSimplifyProper,skills_NumberArithmeticAddConcept,skills_NumberArithmeticAdditionDecimal,skills_NumberArithmeticAdditionInteger,skills_NumberArithmeticMultiplicationDecimal,skills_NumberArithmeticMultiplicationInteger,skills_NumberArithmeticPower,positive_skills_AlgebraAddingLikeTerms,positive_skills_EquationsBalanceMethodArrangeTerms,positive_skills_EquationsBalanceMethodDivideCoefficient,positive_skills_EquationsOperationsAdd,positive_skills_EquationsOperationsMultiply,positive_skills_ExpandSingleBrackets,positive_skills_FractionArithmeticAddProperCommonDenominator,positive_skills_FractionArithmeticAddProperUnlikeDenominator,positive_skills_FractionArithmeticMultiplicationProper,positive_skills_FractionArithmeticSimilarImproperToMixed,positive_skills_FractionArithmeticSimilarMixedToImproper,positive_skills_FractionArithmeticSimplifyProper,negative_skills_AlgebraAddingLikeTerms,negative_skills_AlgebraProductOfPowerFunctions,negative_skills_ApplyBalanceMethodFactor,negative_skills_ArithmeticOrderDivision,negative_skills_ArithmeticOrderMultiplication,negative_skills_EquationsOperationsAdd,negative_skills_EquationsOperationsMultiply,negative_skills_ExpandSingleBrackets,negative_skills_FactorSingleBrackets,negative_skills_FractionArithmeticAddProperCommonDenominator,negative_skills_FractionArithmeticAddProperUnlikeDenominator,negative_skills_FractionArithmeticEquivalentConcept,negative_skills_FractionArithmeticMultiplicationMixed,negative_skills_FractionArithmeticMultiplicationProper,negative_skills_FractionArithmeticSimilarImproperToMixed,negative_skills_FractionArithmeticSimilarMixed,negative_skills_FractionArithmeticSimilarMixedToImproper,negative_skills_NumberArithmeticAddConcept,negative_skills_NumberArithmeticAdditionDecimal,negative_skills_NumberArithmeticAdditionInteger,negative_skills_NumberArithmeticMultiplicationDecimal,negative_skills_NumberArithmeticMultiplicationInteger,negative_skills_NumberArithmeticPower
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[CALCULATE],False,1.00,4.0,Q1,QUESTION,Ex8wE,INTERACTION,MULTISTEP,n =4 c ^{3},n = 4\cdot c \cdot c \cdot c,[EVALUATE],[FINISHED],FINISHED,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[CALCULATE],False,1.00,4.0,Q2,QUESTION,gBslt,INTERACTION,MULTISTEP,u =\frac{4}{25} t ^{3},u = t \cdot t \cdot t \cdot \frac{2}{5}\cdot \...,"[EVALUATE, EVALUATE, HINT, 

In [36]:
df_onehot.to_csv('algebrakit_firstbatch_2_onehotencoding.csv')

In [37]:
df_on = pd.read_csv("algebrakit_firstbatch_2_onehotencoding.csv")
df_on.head()

,Unnamed: 0,exerciseID,sessionID,success,creationTimestamp,stepTypes,finished,marksEarned,marksTotal,questionID,questionType,interactionID,itemType,interactionType,solution,content,event_types,event_evaluations,result_status,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors,skills_AlgebraAddingLikeTerms,skills_AlgebraProductOfPowerFunctions,skills_ArithmeticOrderDivision,skills_ArithmeticOrderMultiplication,skills_EquationsBalanceMethodArrangeTerms,skills_EquationsBalanceMethodDivideCoefficient,skills_EquationsOperationsAdd,skills_EquationsOperationsMultiply,skills_ExpandSingleBrackets,skills_FactorSingleBrackets,skills_FractionArithmeticAddProperCommonDenominator,skills_FractionArithmeticAddProperUnlikeDenominator,skills_FractionArithmeticDivisionProper,skills_FractionArithmeticEquivalentConcept,skills_FractionArithmeticMultiplicationMixed,skills_FractionArithmeticMultiplicationProper,skills_FractionArithmeticSimilarImproperToMixed,skills_FractionArithmeticSimilarMixed,skills_FractionArithmeticSimilarMixedToImproper,skills_FractionArithmeticSimplifyProper,skills_NumberArithmeticAddConcept,skills_NumberArithmeticAdditionDecimal,skills_NumberArithmeticAdditionInteger,skills_NumberArithmeticMultiplicationDecimal,skills_NumberArithmeticMultiplicationInteger,skills_NumberArithmeticPower,positive_skills_AlgebraAddingLikeTerms,positive_skills_EquationsBalanceMethodArrangeTerms,positive_skills_EquationsBalanceMethodDivideCoefficient,positive_skills_EquationsOperationsAdd,positive_skills_EquationsOperationsMultiply,positive_skills_ExpandSingleBrackets,positive_skills_FractionArithmeticAddProperCommonDenominator,positive_skills_FractionArithmeticAddProperUnlikeDenominator,positive_skills_FractionArithmeticMultiplicationProper,positive_skills_FractionArithmeticSimilarImproperToMixed,positive_skills_FractionArithmeticSimilarMixedToImproper,positive_skills_FractionArithmeticSimplifyProper,negative_skills_AlgebraAddingLikeTerms,negative_skills_AlgebraProductOfPowerFunctions,negative_skills_ApplyBalanceMethodFactor,negative_skills_ArithmeticOrderDivision,negative_skills_ArithmeticOrderMultiplication,negative_skills_EquationsOperationsAdd,negative_skills_EquationsOperationsMultiply,negative_skills_ExpandSingleBrackets,negative_skills_FactorSingleBrackets,negative_skills_FractionArithmeticAddProperCommonDenominator,negative_skills_FractionArithmeticAddProperUnlikeDenominator,negative_skills_FractionArithmeticEquivalentConcept,negative_skills_FractionArithmeticMultiplicationMixed,negative_skills_FractionArithmeticMultiplicationProper,negative_skills_FractionArithmeticSimilarImproperToMixed,negative_skills_FractionArithmeticSimilarMixed,negative_skills_FractionArithmeticSimilarMixedToImproper,negative_skills_NumberArithmeticAddConcept,negative_skills_NumberArithmeticAdditionDecimal,negative_skills_NumberArithmeticAdditionInteger,negative_skills_NumberArithmeticMultiplicationDecimal,negative_skills_NumberArithmeticMultiplicationInteger,negative_skills_NumberArithmeticPower
0,0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,['CALCULATE'],False,1.00,4.0,Q1,QUESTION,Ex8wE,INTERACTION,MULTISTEP,n =4 c ^{3},n = 4\cdot c \cdot c \cdot c,['EVALUATE'],['FINISHED'],FINISHED,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,['CALCULATE'],False,1.00,4.0,Q2,QUESTION,gBslt,INTERACTION,MULTISTEP,u =\frac{4}{25} t ^{3},u = t \cdot t \cdot t \cdot \frac{2}{5}\cdot \...,"['EV